# ConvNeXt Tiny Model for Pen Classification
Training ConvNeXt Tiny model for pen identification task

In [33]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

from tqdm import tqdm

## Load and Split Dataset

In [ ]:
df = pd.read_csv("../train.csv")

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df["writer_id"]))

train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]

print(len(train_df), len(val_df))

33200 7050


## Define Dataset Class

In [35]:
class PenDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        
        image = Image.open(img_path).convert("RGB")   # grayscale
        label = int(row["pen_id"]) - 1  # 0-based
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

## Define Data Transforms

In [36]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomApply([
        transforms.RandomAffine(
            degrees=0,
            translate=(0.05, 0.05)
        )
    ], p=0.5),
    transforms.ColorJitter(contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Create DataLoaders

In [ ]:
root_dir = "../"

train_dataset = PenDataset(train_df, root_dir, train_transform)
val_dataset = PenDataset(val_df, root_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=8)

## Setup Device

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Initialize ConvNeXt Tiny Model

In [39]:
model = models.convnext_tiny(weights="ConvNeXt_Tiny_Weights.DEFAULT")

# ConvNeXt models have classifier: [LayerNorm2d, Flatten, Linear]
in_features = model.classifier[2].in_features

model.classifier = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(in_features, 8)
)

model = model.to(device)

## Setup Loss and Optimizer

In [40]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=2e-4, weight_decay=5e-4)

## Setup Learning Rate Scheduler

In [41]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10   # number of epochs for one cycle
)

## Define Training Function

In [42]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    return total_loss / len(loader), acc

## Define Evaluation Function

In [43]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro")
    rec = recall_score(all_labels, all_preds, average="macro")

    return total_loss / len(loader), acc, prec, rec

## Train the Model

In [44]:
num_epochs = 20
patience = 3

best_val_loss = float("inf")
counter = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_prec, val_rec = evaluate(model, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Precision: {val_prec:.4f} | Val Recall: {val_rec:.4f}")
    scheduler.step()
    
    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), "best_model_convnext_tiny.pth")
        print("✅ Best model saved")

    else:
        counter += 1
        print(f"Early stopping counter: {counter}/{patience}")

        if counter >= patience:
            print("⛔ Early stopping triggered")
            break


Epoch 1/20


  0%|          | 0/260 [00:00<?, ?it/s]

  0%|          | 0/260 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 69.50 MiB is free. Process 2787 has 266.07 MiB memory in use. Process 2320848 has 18.97 GiB memory in use. Including non-PyTorch memory, this process has 25.93 GiB memory in use. Of the allocated memory 25.46 GiB is allocated by PyTorch, and 163.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Inference on Test Set

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os
from tqdm import tqdm
import torch.nn as nn

# -----------------------
# Load test CSV
# -----------------------
test_df = pd.read_csv("../test.csv")

# -----------------------
# Dataset (no labels)
# -----------------------
class TestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, row["image_id"]

# -----------------------
# Transform (same as val)
# -----------------------
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -----------------------
# DataLoader
# -----------------------
test_dataset = TestDataset(test_df, "", test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=8)

# -----------------------
# Load model
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights="ConvNeXt_Tiny_Weights.DEFAULT")

in_features = model.classifier[2].in_features

model.classifier = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(in_features, 8)
)

model.load_state_dict(torch.load("best_model_convnext_tiny.pth", map_location=device))
model = model.to(device)
model.eval()

## Make Predictions with TTA (Test Time Augmentation)

In [ ]:
# -----------------------
# Inference
# -----------------------
predictions = []
image_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader):
        images = images.to(device)

        outputs = model(images)
        flipped_images = torch.flip(images, dims=[3])
        outputs_flipped = model(flipped_images)

        flipped_v = torch.flip(images, dims=[2])
        outputs_v = model(flipped_v)

        outputs = (outputs + outputs_flipped + outputs_v) / 3
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())
        image_ids.extend(ids)

# -----------------------
# Convert back to 1-based labels
# -----------------------
predictions = [p + 1 for p in predictions]

# -----------------------
# Create submission
# -----------------------
submission = pd.DataFrame({
    "image_id": image_ids,
    "pen_id": predictions
})

submission.to_csv("submission_convnext_tiny.csv", index=False)

print("✅ submission_convnext_tiny.csv saved")

## Analysis - Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

def get_predictions(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_labels, all_preds

In [ ]:
y_true, y_pred = get_predictions(model, val_loader, device)

cm = confusion_matrix(y_true, y_pred)

print(cm)

## Visualize Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - ConvNeXt Tiny")
plt.show()